# Tiled Matrix Multiplication

Build matrix multiplication from scratch on the GPU: start with a naive kernel, analyze its memory bottleneck, then implement the tiled algorithm using shared memory for dramatic speedups. The cornerstone of GPU computing.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-tiling-matmul)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Naive Matrix Multiplication

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda

# ============================================
# Naive Matrix Multiplication
# ============================================

@cuda.jit
def naive_matmul(A, B, C):
    """Each thread computes one element of C."""
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        total = 0.0
        for k in range(A.shape[1]):
            total += A[row, k] * B[k, col]
        C[row, col] = total

# Test correctness first with small matrices
print("=== Correctness Test ===")
M, K, N = 64, 128, 96
A = np.random.randn(M, K).astype(np.float32)
B = np.random.randn(K, N).astype(np.float32)
C_expected = A @ B

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((M, N), dtype=np.float32)

TPB = (16, 16)
BPG = ((N + TPB[0] - 1) // TPB[0], (M + TPB[1] - 1) // TPB[1])

naive_matmul[BPG, TPB](d_A, d_B, d_C)
C_gpu = d_C.copy_to_host()

assert np.allclose(C_gpu, C_expected, atol=1e-3), "Results mismatch!"
print(f"  {M}x{K} @ {K}x{N} = {M}x{N}: PASSED")

# Benchmark at different sizes
print("\n=== Naive Matmul Benchmark ===")
sizes = [256, 512, 1024, 2048]

for size in sizes:
    M = K = N = size
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)

    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)

    BPG = ((N + TPB[0] - 1) // TPB[0], (M + TPB[1] - 1) // TPB[1])

    # Warm up
    naive_matmul[BPG, TPB](d_A, d_B, d_C)
    cuda.synchronize()

    # Benchmark
    iters = 10 if size <= 1024 else 3
    start = time.perf_counter()
    for _ in range(iters):
        naive_matmul[BPG, TPB](d_A, d_B, d_C)
    cuda.synchronize()
    gpu_time = (time.perf_counter() - start) / iters

    # Calculate GFLOPS: 2*M*N*K FLOPs for matmul
    flops = 2 * M * N * K
    gflops = flops / gpu_time / 1e9

    # Compare with NumPy
    start = time.perf_counter()
    for _ in range(iters):
        C_cpu = A @ B
    cpu_time = (time.perf_counter() - start) / iters

    print(f"  {size}x{size}: GPU={gpu_time*1000:.1f}ms ({gflops:.0f} GFLOPS), "
          f"NumPy={cpu_time*1000:.1f}ms, Speedup={cpu_time/gpu_time:.1f}x")

    # Verify correctness
    assert np.allclose(d_C.copy_to_host(), A @ B, atol=1e-2 * size/256)

print("\nNote: Naive matmul achieves very low GFLOPS because it is")
print("memory-bound. Each element of A and B is loaded from global")
print("memory K times. The tiled version in the next section fixes this.")

# ============================================
# Memory Access Analysis
# ============================================
print("\n=== Memory Access Analysis (1024x1024) ===")
size = 1024
total_output_elements = size * size
reads_per_element = 2 * size  # K reads from A + K reads from B
total_reads = total_output_elements * reads_per_element
unique_elements = 2 * size * size
redundancy = total_reads / unique_elements

print(f"  Output elements: {total_output_elements:,}")
print(f"  Global reads per element: {reads_per_element:,}")
print(f"  Total global reads: {total_reads:,} ({total_reads * 4 / 1e9:.1f} GB)")
print(f"  Unique data: {unique_elements:,} elements ({unique_elements * 4 / 1e6:.1f} MB)")
print(f"  Redundancy: {redundancy:.0f}x (each element read {redundancy:.0f} times!)")
print(f"  Arithmetic intensity: {2*size / (2*size*4):.2f} FLOPs/byte")
print(f"  T4 balance point: ~25 FLOPs/byte")
print(f"  -> We are {25 / (2*size / (2*size*4)):.0f}x below the balance point!")

### Lesson example: Tiling Strategy

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda, float32

# ============================================
# Tiling Visualization: Phase-by-Phase
# ============================================

print("=== Tiling Strategy Visualization ===")
print("\nComputing C = A @ B where A is MxK, B is KxN, C is MxN")
print("With TILE_SIZE = 16:\n")

M, K, N = 64, 48, 64
TILE = 16
num_phases = (K + TILE - 1) // TILE

print(f"Matrix sizes: A={M}x{K}, B={K}x{N}, C={M}x{N}")
print(f"TILE_SIZE = {TILE}")
print(f"Number of phases = ceil({K}/{TILE}) = {num_phases}")
print(f"Blocks in grid: ({N//TILE}, {M//TILE}) = ({N//TILE}x{M//TILE} blocks)")
print(f"\nFor block (0,0) computing C[0:16, 0:16]:")
for p in range(num_phases):
    a_col_start = p * TILE
    a_col_end = min(a_col_start + TILE, K)
    b_row_start = p * TILE
    b_row_end = min(b_row_start + TILE, K)
    print(f"  Phase {p}: Load A[0:16, {a_col_start}:{a_col_end}] and B[{b_row_start}:{b_row_end}, 0:16] into shared memory")
    print(f"           Compute partial dot products, accumulate into running sum")

# ============================================
# Memory Access Comparison
# ============================================
print(f"\n=== Memory Access Analysis (1024x1024) ===")
size = 1024
TILE = 16

# Naive
naive_reads_per_block = TILE * TILE * 2 * size  # Each of TILE^2 threads reads 2*K
naive_total = (size // TILE) * (size // TILE) * naive_reads_per_block

# Tiled
phases = size // TILE
tiled_reads_per_block = phases * 2 * TILE * TILE  # Each phase: 2 tiles of TILE^2
tiled_total = (size // TILE) * (size // TILE) * tiled_reads_per_block

print(f"  Naive global reads:  {naive_total:>15,} ({naive_total * 4 / 1e9:.1f} GB)")
print(f"  Tiled global reads:  {tiled_total:>15,} ({tiled_total * 4 / 1e9:.1f} GB)")
print(f"  Reduction: {naive_total / tiled_total:.0f}x fewer global reads")
print(f"  (= TILE_SIZE = {TILE})")

# ============================================
# Simple tiled matmul to demonstrate concept
# ============================================
print(f"\n=== Tiled Matmul (TILE_SIZE={TILE}) ===")

TILE_SIZE = 16

@cuda.jit
def tiled_matmul_simple(A, B, C):
    """Tiled matrix multiplication (assumes dimensions are multiples of TILE_SIZE)."""
    # Shared memory tiles
    As = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float32)
    Bs = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    # Output position
    row = cuda.blockIdx.y * TILE_SIZE + ty
    col = cuda.blockIdx.x * TILE_SIZE + tx

    total = 0.0
    # Loop over tiles
    for phase in range(A.shape[1] // TILE_SIZE):
        # Collaboratively load tile from A and B
        As[ty, tx] = A[row, phase * TILE_SIZE + tx]
        Bs[ty, tx] = B[phase * TILE_SIZE + ty, col]
        cuda.syncthreads()  # Ensure tile is loaded

        # Compute partial dot product from this tile
        for k in range(TILE_SIZE):
            total += As[ty, k] * Bs[k, tx]
        cuda.syncthreads()  # Ensure all threads done before next tile

    C[row, col] = total

# Test: dimensions must be multiples of TILE_SIZE for this simple version
M, K, N = 512, 512, 512
A = np.random.randn(M, K).astype(np.float32)
B = np.random.randn(K, N).astype(np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((M, N), dtype=np.float32)

TPB = (TILE_SIZE, TILE_SIZE)
BPG = (N // TILE_SIZE, M // TILE_SIZE)

tiled_matmul_simple[BPG, TPB](d_A, d_B, d_C)
C_gpu = d_C.copy_to_host()

assert np.allclose(C_gpu, A @ B, atol=1e-2), "Tiled matmul results incorrect!"
print(f"  {M}x{K} @ {K}x{N}: Correct!")

# Compare naive vs tiled
@cuda.jit
def naive_matmul(A, B, C):
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        total = 0.0
        for k in range(A.shape[1]):
            total += A[row, k] * B[k, col]
        C[row, col] = total

for size in [512, 1024]:
    M = K = N = size
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    d_A, d_B = cuda.to_device(A), cuda.to_device(B)
    d_C_naive = cuda.device_array((M, N), dtype=np.float32)
    d_C_tiled = cuda.device_array((M, N), dtype=np.float32)

    BPG_naive = ((N + 15) // 16, (M + 15) // 16)
    BPG_tiled = (N // TILE_SIZE, M // TILE_SIZE)

    # Warm up
    naive_matmul[BPG_naive, TPB](d_A, d_B, d_C_naive)
    tiled_matmul_simple[BPG_tiled, TPB](d_A, d_B, d_C_tiled)
    cuda.synchronize()

    iters = 10
    start = time.perf_counter()
    for _ in range(iters):
        naive_matmul[BPG_naive, TPB](d_A, d_B, d_C_naive)
    cuda.synchronize()
    naive_time = (time.perf_counter() - start) / iters

    start = time.perf_counter()
    for _ in range(iters):
        tiled_matmul_simple[BPG_tiled, TPB](d_A, d_B, d_C_tiled)
    cuda.synchronize()
    tiled_time = (time.perf_counter() - start) / iters

    flops = 2 * M * N * K
    print(f"\n  {size}x{size} matmul:")
    print(f"    Naive:  {naive_time*1000:.1f} ms ({flops/naive_time/1e9:.0f} GFLOPS)")
    print(f"    Tiled:  {tiled_time*1000:.1f} ms ({flops/tiled_time/1e9:.0f} GFLOPS)")
    print(f"    Speedup: {naive_time/tiled_time:.1f}x")

### Lesson example: Complete Tiled Matrix Multiplication

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda, float32

# ============================================
# Complete Tiled Matrix Multiplication
# ============================================

TILE_SIZE = 16

@cuda.jit
def naive_matmul(A, B, C):
    """Naive: one thread per output element, reads all from global memory."""
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        total = 0.0
        for k in range(A.shape[1]):
            total += A[row, k] * B[k, col]
        C[row, col] = total

@cuda.jit
def tiled_matmul(A, B, C):
    """Tiled: uses shared memory to reduce global memory traffic by TILE_SIZE."""
    As = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float32)
    Bs = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    row = cuda.blockIdx.y * TILE_SIZE + ty
    col = cuda.blockIdx.x * TILE_SIZE + tx

    M = C.shape[0]
    N = C.shape[1]
    K = A.shape[1]

    total = 0.0
    num_phases = (K + TILE_SIZE - 1) // TILE_SIZE

    for phase in range(num_phases):
        # Load tiles with boundary checking
        a_col = phase * TILE_SIZE + tx
        b_row = phase * TILE_SIZE + ty

        if row < M and a_col < K:
            As[ty, tx] = A[row, a_col]
        else:
            As[ty, tx] = 0.0

        if b_row < K and col < N:
            Bs[ty, tx] = B[b_row, col]
        else:
            Bs[ty, tx] = 0.0

        cuda.syncthreads()

        # Partial dot product
        for k in range(TILE_SIZE):
            total += As[ty, k] * Bs[k, tx]

        cuda.syncthreads()

    if row < M and col < N:
        C[row, col] = total

# ============================================
# Test with non-standard dimensions
# ============================================
print("=== Correctness Tests ===")
test_cases = [
    (64, 64, 64),
    (100, 200, 150),     # Non-multiples of 16
    (1000, 1500, 1200),  # Larger, non-multiples
    (1024, 1024, 1024),  # Power of 2
]

for M, K, N in test_cases:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    expected = A @ B

    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)

    TPB = (TILE_SIZE, TILE_SIZE)
    BPG = ((N + TILE_SIZE - 1) // TILE_SIZE, (M + TILE_SIZE - 1) // TILE_SIZE)

    tiled_matmul[BPG, TPB](d_A, d_B, d_C)
    result = d_C.copy_to_host()

    # Tolerance scales with K (more accumulations = more float error)
    atol = 1e-2 * (K / 64)
    assert np.allclose(result, expected, atol=atol), f"Failed for {M}x{K} @ {K}x{N}"
    print(f"  {M}x{K} @ {K}x{N} = {M}x{N}: PASSED (atol={atol:.3f})")

# ============================================
# Performance Comparison: Naive vs Tiled vs NumPy
# ============================================
print("\n=== Performance Comparison ===")
print(f"{'Size':>8} | {'Naive (ms)':>11} | {'Tiled (ms)':>11} | {'NumPy (ms)':>11} | {'Naive GFLOPS':>13} | {'Tiled GFLOPS':>13} | {'Tiled/Naive':>12}")
print("-" * 100)

for size in [256, 512, 1024, 2048]:
    M = K = N = size
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)

    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C_naive = cuda.device_array((M, N), dtype=np.float32)
    d_C_tiled = cuda.device_array((M, N), dtype=np.float32)

    TPB = (TILE_SIZE, TILE_SIZE)
    BPG = ((N + TILE_SIZE - 1) // TILE_SIZE, (M + TILE_SIZE - 1) // TILE_SIZE)

    # Warm up
    naive_matmul[BPG, TPB](d_A, d_B, d_C_naive)
    tiled_matmul[BPG, TPB](d_A, d_B, d_C_tiled)
    cuda.synchronize()

    flops = 2.0 * M * N * K
    iters = 10 if size <= 1024 else 3

    # Naive
    start = time.perf_counter()
    for _ in range(iters):
        naive_matmul[BPG, TPB](d_A, d_B, d_C_naive)
    cuda.synchronize()
    naive_t = (time.perf_counter() - start) / iters

    # Tiled
    start = time.perf_counter()
    for _ in range(iters):
        tiled_matmul[BPG, TPB](d_A, d_B, d_C_tiled)
    cuda.synchronize()
    tiled_t = (time.perf_counter() - start) / iters

    # NumPy (CPU)
    start = time.perf_counter()
    for _ in range(iters):
        _ = A @ B
    numpy_t = (time.perf_counter() - start) / iters

    naive_gf = flops / naive_t / 1e9
    tiled_gf = flops / tiled_t / 1e9

    print(f"{size:>8} | {naive_t*1000:>11.1f} | {tiled_t*1000:>11.1f} | {numpy_t*1000:>11.1f} | {naive_gf:>13.0f} | {tiled_gf:>13.0f} | {naive_t/tiled_t:>11.1f}x")

print("\nKey takeaway: Tiling gives 4-7x speedup over naive by reducing")
print(f"global memory reads by {TILE_SIZE}x (TILE_SIZE={TILE_SIZE}).")
print("cuBLAS (used by NumPy on GPU) is even faster due to register")
print("tiling, Tensor Cores, and other advanced optimizations.")

---

## Exercise: Tiled Matrix Multiplication: Implement and Benchmark


In [ ]:
import numpy as np
import time
from numba import cuda, float32

TILE_SIZE = 16

# TODO: Implement naive matrix multiplication
@cuda.jit
def naive_matmul(A, B, C):
    """Each thread computes one element of C = A @ B."""
    pass

# TODO: Implement tiled matrix multiplication
@cuda.jit
def tiled_matmul(A, B, C):
    """Tiled matmul using shared memory. Handles arbitrary dimensions."""
    # Hint: Allocate shared memory
    # As = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float32)
    # Bs = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float32)
    pass

def benchmark_matmul(kernel, M, K, N, iters=10):
    """Benchmark a matmul kernel and return time in seconds."""
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)

    TPB = (TILE_SIZE, TILE_SIZE)
    BPG = ((N + TILE_SIZE - 1) // TILE_SIZE, (M + TILE_SIZE - 1) // TILE_SIZE)

    # Warm up
    kernel[BPG, TPB](d_A, d_B, d_C)
    cuda.synchronize()

    # TODO: Time the kernel over multiple iterations
    # TODO: Return average time, and verify against A @ B
    pass

# TODO: Test correctness with non-aligned dimensions (e.g., 1000x1500 @ 1500x1200)
# TODO: Benchmark at sizes [256, 512, 1024, 2048]
# TODO: Print results table with GFLOPS and speedup

## Your tasks

Implement the tiled matrix multiplication kernel from scratch, test it with various matrix sizes, and benchmark it against the naive version.

**Requirements:**
1. Implement `naive_matmul(A, B, C)` -- one thread per output element, inner loop over K
2. Implement `tiled_matmul(A, B, C)` with:
   - TILE_SIZE = 16
   - Shared memory tiles for A and B
   - Proper boundary handling (works for any matrix size)
   - Two syncthreads() per phase
3. Test both with non-square, non-tile-aligned matrices (e.g., 1000x1500 @ 1500x1200)
4. Benchmark at sizes 256, 512, 1024, 2048 (square matrices)
5. Report GFLOPS for each version and compute the speedup
6. Verify the tiled result matches NumPy's A @ B

**Hints:**
- Shared memory: `cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float32)`
- Boundary check: load 0.0 for out-of-bounds elements
- Number of phases: `(K + TILE_SIZE - 1) // TILE_SIZE`
- GFLOPS = 2 * M * N * K / time / 1e9
- The tolerance for `np.allclose` should scale with K (more additions = more float error)
- Remember: BPG = ((N + T - 1) // T, (M + T - 1) // T) where x=cols, y=rows

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-tiling-matmul) for the solution and discussion.